# MMAudio ile Ses Üretimi

**MMAudio** video ve/veya metin girdisinden ses üreten bir yapay zeka modelidir.

Bu notebook'ta üç mod desteklenir:
- **Video + Text → Audio** — Bir video yükleyip metin ile yönlendirin (en iyi kalite)
- **Video → Audio** — Sadece video verin, model sesi otomatik üretsin
- **Text → Audio** — Sadece metin açıklaması ile ses üretin

**Alt klasör desteği:** Video klasörünüz alt klasörler içerebilir — çıktıda aynı klasör yapısı korunur.

Tüm parametreler **Konfigürasyon** hücresinde toplanmıştır — farklı değerlerle deney yapabilirsiniz.

**Model:** `large_44k_v2` (44kHz, yüksek kalite) veya `small_16k` (16kHz, hızlı)  
**Lisans:** Model ağırlıkları CC-BY-NC 4.0 (yalnızca ticari olmayan kullanım)  
**GitHub:** https://github.com/hkchengrex/MMAudio

## GPU Ayarı

Bu notebook GPU gerektirir. T4 GPU (ücretsiz) yeterlidir.

1. Üst menüden **Runtime** → **Change runtime type** tıklayın
2. **Hardware accelerator** altından **T4 GPU** seçin
3. **Save** tıklayın

Aşağıdaki hücreyi çalıştırarak GPU'nun bağlı olduğunu doğrulayın.

In [1]:
# GPU bagli mi kontrol et
import torch

if torch.cuda.is_available():
    # GPU adini ve bellek bilgisini yazdir
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    gpu_memory = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / (1024**3)
    print(f"GPU bulundu: {gpu_name}")
    print(f"GPU bellegi: {gpu_memory:.1f} GB")
else:
    # GPU yoksa uyari ver
    print("UYARI: GPU bulunamadi!")
    print("Lutfen Runtime > Change runtime type > T4 GPU secin.")
    print("GPU olmadan model cok yavas calisir.")

GPU bulundu: Tesla T4
GPU bellegi: 14.6 GB


## Konfigürasyon

Aşağıdaki hücrede tüm ayarlanabilir parametreler bir arada verilmiştir.  
Değerleri değiştirip tekrar çalıştırarak farklı sonuçları deneyebilirsiniz.

### Google Drive Ayarları

Videolarınızı Google Drive'da bir klasöre yükleyin. Notebook oradan okuyup sonuçları output klasörüne kaydeder.  
**Alt klasörler desteklenir** — klasör yapısı çıktıda aynen korunur.

| Parametre | Default | Açıklama |
|-----------|---------|----------|
| **DRIVE_VIDEO_FOLDER** | `"mmaudio-videos"` | Drive'daki video klasörü (MyDrive altında) |
| **DRIVE_OUTPUT_FOLDER** | `"mmaudio-outputs"` | Sonuçların kaydedileceği klasör |
| **BATCH_MODE** | `False` | `True` = tüm videoları sırayla işle |
| **VIDEO_INDEX** | `0` | `BATCH_MODE=False` ise hangi videoyu işle (listeden index) |

### Model Cache Ayarları

Model dosyaları (~2 GB) ilk çalıştırmada indirilir. Drive'da cache'leyerek sonraki seferlerde indirmeyi atlayabilirsiniz.

| Parametre | Default | Açıklama |
|-----------|---------|----------|
| **CACHE_MODELS_ON_DRIVE** | `True` | `True` = model dosyalarını Drive'da cachele (sonraki seferlerde indirme atlanır) |
| **DRIVE_MODEL_FOLDER** | `"mmaudio-models"` | Model cache klasörü (MyDrive altında) |

### Üretim Ayarları

Ses süresi otomatik olarak videonun süresine göre ayarlanır.

| Parametre | Default | Açıklama |
|-----------|---------|----------|
| **PROMPT** | — | Üretmek istediğiniz sesi tarif edin (İngilizce). Video modunda opsiyonel rehber |
| **NEGATIVE_PROMPT** | `""` | İstemediğiniz sesleri tarif edin (ör. `"music, speech"`) |
| **MODEL_NAME** | `"large_44k"` | Model seçimi (NSFW fine-tuned `"large_44k"` mimarisi) |
| **SEED** | `42` | Rastgele sayı tohumu. Aynı seed = aynı sonuç (tekrarlanabilirlik) |
| **NUM_STEPS** | `25` | Flow matching adım sayısı. Fazla = kaliteli ama yavaş, az = hızlı ama kaba |
| **INFERENCE_MODE** | `"euler"` | `"euler"` (sabit adım, hızlı) veya `"adaptive"` (otomatik adım, daha doğru) |
| **CFG_STRENGTH** | `4.5` | Rehberlik gücü. Yüksek = prompt'a daha sadık, düşük = daha serbest |

### Çıktı İsimlendirmesi (alt klasör yapısı korunur)

```
mmaudio-videos/dog_running.mp4           →  mmaudio-outputs/dog_running_audio.mp4
mmaudio-videos/scene1/clip_a.mp4         →  mmaudio-outputs/scene1/clip_a_audio.mp4
mmaudio-videos/scene1/subdir/deep.mp4    →  mmaudio-outputs/scene1/subdir/deep_audio.mp4
```

In [2]:
# ============================================================
#  KONFIGURASYON - Tum ayarlari buradan degistirin
# ============================================================

# --- Google Drive ---
# Drive'daki video klasoru (MyDrive altinda)
DRIVE_VIDEO_FOLDER = "mmaudio-videos"

# Sonuclarin kaydedilecegi klasor (MyDrive altinda)
DRIVE_OUTPUT_FOLDER = "mmaudio-outputs"

# True = tum videolari sirayla isle, False = sadece VIDEO_INDEX'teki videoyu isle
BATCH_MODE = True

# BATCH_MODE=False ise hangi videoyu isle (0'dan baslar)
VIDEO_INDEX = 0

# --- Model Cache ---
# True = model dosyalarini Drive'da cachele (sonraki seferlerde indirme atlanir)
# False = her seferde HuggingFace'den indir
CACHE_MODELS_ON_DRIVE = True

# Model cache klasoru (MyDrive altinda)
DRIVE_MODEL_FOLDER = "mmaudio-models"

# --- Video Chunking ---
# Bu sureden uzun videolar parcalara ayrilir (saniye)
# Her parca MAX/2 ile MAX arasi olur (ornek: 7-14s)
MAX_CHUNK_DURATION = 14

# --- Prompt ---
# Sesi tarif edin (Ingilizce). Video modunda opsiyonel rehber olarak kullanilir
PROMPT = ""

# Istemediginiz sesleri tarif edin (bos birakabilirsiniz)
# Ornek: "music, speech, noise"
NEGATIVE_PROMPT = ""

# Rastgele sayi tohumu - ayni seed ayni sonucu verir (tekrarlanabilirlik)
SEED = 42

# --- Model ---
# NSFW fine-tuned model "large_44k" (v1) mimarisini kullanir, degistirmeyin
MODEL_NAME = "large_44k"

# --- Flow Matching ---
# Adim sayisi: fazla = kaliteli ama yavas, az = hizli ama kaba
# Onerilen aralik: 10-50, default: 25
NUM_STEPS = 25

# Cozucu modu:
#   "euler"    = sabit adim, hizli ve tahmin edilebilir
#   "adaptive" = otomatik adim boyutu, daha dogru (NUM_STEPS gormezden gelinir)
INFERENCE_MODE = "euler"

# --- Rehberlik ---
# Classifier-free guidance gucu
# Yuksek = prompt'a daha sadik, dusuk = daha serbest
# Onerilen aralik: 1.0-10.0, default: 4.5
CFG_STRENGTH = 4.5

# ============================================================
print("=== Konfigurasyon ===")
print(f"  Drive Video:     {DRIVE_VIDEO_FOLDER}/")
print(f"  Drive Output:    {DRIVE_OUTPUT_FOLDER}/")
print(f"  Batch Mode:      {BATCH_MODE}")
if not BATCH_MODE:
    print(f"  Video Index:     {VIDEO_INDEX}")
if CACHE_MODELS_ON_DRIVE:
    print(f"  Model Cache:     Drive/{DRIVE_MODEL_FOLDER}/ (acik)")
else:
    print(f"  Model Cache:     kapali (her seferde indirir)")
print(f"  Max Chunk Sure:  {MAX_CHUNK_DURATION}s (uzun videolar parcalanir)")
print(f"  Prompt:          {PROMPT or '(bos)'}")
print(f"  Negatif Prompt:  {NEGATIVE_PROMPT or '(yok)'}")
print(f"  Sure:            videonun suresi")
print(f"  Seed:            {SEED}")
print(f"  Model:           {MODEL_NAME} (NSFW fine-tuned)")
print(f"  Adim Sayisi:     {NUM_STEPS}")
print(f"  Cozucu:          {INFERENCE_MODE}")
print(f"  CFG Gucu:        {CFG_STRENGTH}")
print("=" * 25)

=== Konfigurasyon ===
  Drive Video:     mmaudio-videos/
  Drive Output:    mmaudio-outputs/
  Batch Mode:      True
  Model Cache:     Drive/mmaudio-models/ (acik)
  Max Chunk Sure:  14s (uzun videolar parcalanir)
  Prompt:          (bos)
  Negatif Prompt:  (yok)
  Sure:            videonun suresi
  Seed:            42
  Model:           large_44k (NSFW fine-tuned)
  Adim Sayisi:     25
  Cozucu:          euler
  CFG Gucu:        4.5


## Google Drive Bağlantısı

Önce Drive izinlerini alıp video klasörünü tarayalım — böylece sorun varsa uzun kurulum adımlarını beklemeye gerek kalmaz.  
İlk çalıştırmada Google hesabınıza erişim izni isteyecektir.

In [3]:
# Google Drive'i bagla ve video klasorunu recursive tara
from google.colab import drive
import os

drive.mount('/content/drive')

drive_video_path = f"/content/drive/MyDrive/{DRIVE_VIDEO_FOLDER}"
drive_output_path = f"/content/drive/MyDrive/{DRIVE_OUTPUT_FOLDER}"

# Output klasorunu olustur (yoksa)
os.makedirs(drive_output_path, exist_ok=True)

# Video dosyalarini recursive bul — (rel_dir, filename) tuple listesi
# rel_dir: alt klasor yolu (kok icin ""), filename: dosya adi
video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.webm')

def find_videos(base_path, extensions):
    """Klasor agacini tara, (rel_dir, filename) tuple listesi dondur."""
    result = []
    for root, dirs, files in os.walk(base_path):
        dirs.sort()
        for f in sorted(files):
            if f.lower().endswith(extensions):
                rel_dir = os.path.relpath(root, base_path)
                if rel_dir == '.':
                    rel_dir = ''
                result.append((rel_dir, f))
    return result

video_items = find_videos(drive_video_path, video_extensions)

# Klasor yapisiyla listele
print(f"Klasor: Drive/{DRIVE_VIDEO_FOLDER}/")
print(f"Bulunan videolar ({len(video_items)}):")
prev_dir = None
for i, (rel_dir, fname) in enumerate(video_items):
    if rel_dir != prev_dir:
        if rel_dir:
            print(f"  {rel_dir}/")
        prev_dir = rel_dir
    src_path = f"{drive_video_path}/{rel_dir}/{fname}" if rel_dir else f"{drive_video_path}/{fname}"
    size_mb = os.path.getsize(src_path) / (1024 * 1024)
    indent = "    " if rel_dir else "  "
    print(f"{indent}[{i}] {fname} ({size_mb:.1f} MB)")

print()
if BATCH_MODE:
    print(f"BATCH MODE: Tum {len(video_items)} video islenecek")
else:
    rel_dir, fname = video_items[VIDEO_INDEX]
    display = f"{rel_dir}/{fname}" if rel_dir else fname
    print(f"Secilen video: [{VIDEO_INDEX}] {display}")

Mounted at /content/drive
Klasor: Drive/mmaudio-videos/
Bulunan videolar (151):
  hafta1/1/
    [0] 1.mp4 (1.7 MB)
    [1] 450x_auto__ (10).mp4 (4.3 MB)
    [2] 450x_auto__ (11).mp4 (4.1 MB)
    [3] 450x_auto__ (12).mp4 (5.2 MB)
    [4] 450x_auto__ (13).mp4 (1.2 MB)
    [5] 450x_auto__ (14).mp4 (2.6 MB)
    [6] 450x_auto__ (15).mp4 (1.0 MB)
    [7] 450x_auto__ (29).mp4 (14.8 MB)
    [8] 450x_auto__ (3).mp4 (3.4 MB)
    [9] 450x_auto__ (4).mp4 (9.0 MB)
    [10] 450x_auto__ (5).mp4 (1.7 MB)
    [11] 450x_auto__ (6).mp4 (1.9 MB)
    [12] 450x_auto__ (7).mp4 (3.7 MB)
    [13] 450x_auto__ (8).mp4 (4.1 MB)
    [14] 450x_auto__ (9).mp4 (2.2 MB)
    [15] 450x_auto__.mp4 (2.5 MB)
  hafta1/Gün 2/
    [16] 450x_auto__ (1).mp4 (1.0 MB)
    [17] 450x_auto__ (10).mp4 (2.0 MB)
    [18] 450x_auto__ (11).mp4 (1.2 MB)
    [19] 450x_auto__ (12).mp4 (1.4 MB)
    [20] 450x_auto__ (13).mp4 (0.9 MB)
    [21] 450x_auto__ (14).mp4 (1.0 MB)
    [22] 450x_auto__ (15).mp4 (1.1 MB)
    [23] 450x_auto__ (16).mp4 (

## Yardımcı Fonksiyonlar

Notebook genelinde kullanılan yardımcı fonksiyonlar.  
Bu hücreyi bir kez çalıştırın.

In [4]:
# ============================================================
#  YARDIMCI FONKSIYONLAR
# ============================================================
import os
import gc
import math
import shutil
import subprocess
from pathlib import Path


def _get_fps_mode_args():
    """FFmpeg surumune gore fps_mode veya vsync argumani dondur.
    FFmpeg 5.1+ => ['-fps_mode', 'cfr'], eski => ['-vsync', 'cfr']"""
    try:
        result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
        first_line = result.stdout.splitlines()[0]
        ver_str = first_line.split('version')[1].strip().split('-')[0].split(' ')[0]
        ver_str = ver_str.lstrip('nN')
        major = int(ver_str.split('.')[0])
        minor = int(ver_str.split('.')[1]) if '.' in ver_str else 0
        if major > 5 or (major == 5 and minor >= 1):
            return ['-fps_mode', 'cfr']
    except Exception:
        pass
    return ['-vsync', 'cfr']


_FPS_MODE_ARGS = _get_fps_mode_args()
print(f"FFmpeg CFR argumani: {_FPS_MODE_ARGS}")


def make_drive_path(base, rel_dir, filename=""):
    """rel_dir ve filename'den tam yol olustur. rel_dir bos ise atla."""
    parts = [base]
    if rel_dir:
        parts.append(rel_dir)
    if filename:
        parts.append(filename)
    return "/".join(parts)


def resize_video_720p(input_path, output_path):
    """Videoyu 720p'ye kucult + CFR zorlama. Basarisiz olursa ValueError firlatir."""
    result = subprocess.run([
        'ffmpeg', '-y', '-i', input_path,
        '-vf', 'fps=25,scale=-2:720',
        *_FPS_MODE_ARGS,
        '-c:v', 'libx264', '-preset', 'ultrafast', '-crf', '23',
        '-an', output_path
    ], capture_output=True, text=True)
    if result.returncode != 0:
        raise ValueError(f"ffmpeg resize basarisiz: {result.stderr[-300:]}")
    return output_path


def get_video_duration(video_path):
    """ffprobe ile video suresini saniye olarak dondur. Basarisiz olursa ValueError firlatir."""
    probe = subprocess.run(
        ['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
         '-of', 'default=noprint_wrappers=1:nokey=1', video_path],
        capture_output=True, text=True
    )
    if probe.returncode != 0 or not probe.stdout.strip():
        raise ValueError(f"ffprobe basarisiz ({video_path}): {probe.stderr[-300:]}")
    return float(probe.stdout.strip())


def merge_video_audio(video_path, audio_path, output_path):
    """Orijinal video ile uretilen sesi birlestir. Basarisiz olursa ValueError firlatir."""
    result = subprocess.run([
        'ffmpeg', '-y',
        '-i', video_path,
        '-i', audio_path,
        '-c:v', 'copy',
        '-c:a', 'aac',
        '-shortest',
        output_path
    ], capture_output=True, text=True)
    if result.returncode != 0:
        raise ValueError(f"ffmpeg merge basarisiz: {result.stderr[-300:]}")


def cleanup_files(*paths):
    """Verilen dosyalari sessizce sil (yoksa hata verme)."""
    for p in paths:
        if p and os.path.exists(p):
            os.remove(p)


def calculate_chunks(total_duration, max_dur):
    """Video suresini esit parcalara bol. [(start, duration), ...] dondur.
    Kisa video (<=max_dur) icin tek eleman doner.
    Her parca max_dur/2 ile max_dur arasi olur."""
    if total_duration <= max_dur:
        return [(0, total_duration)]
    n = math.ceil(total_duration / max_dur)
    chunk_dur = total_duration / n
    return [(i * chunk_dur, min(chunk_dur, total_duration - i * chunk_dur)) for i in range(n)]


def extract_chunk_720p(input_path, start, duration, output_path):
    """Orijinalden zaman dilimi kes + 720p resize + CFR zorlama (tek ffmpeg komutu).
    Re-encode ile frame-accurate kesim saglanir."""
    result = subprocess.run([
        'ffmpeg', '-y', '-ss', str(start), '-i', input_path,
        '-t', str(duration), '-vf', 'fps=25,scale=-2:720',
        *_FPS_MODE_ARGS,
        '-c:v', 'libx264', '-preset', 'ultrafast', '-crf', '23',
        '-an', output_path
    ], capture_output=True, text=True)
    if result.returncode != 0:
        raise ValueError(f"ffmpeg chunk extract basarisiz: {result.stderr[-300:]}")



def fix_video_frames(video_info, seq_cfg):
    """clip_frames boyutunu modelin bekledigine esle.
    Kisa ise son frame'i kopyalayarak doldur, uzunsa kirp.
    NOT: sync_frames'e dokunma — sync encoder temporal compression
    uyguladigi icin raw frame sayisi != sync_seq_len."""
    import torch

    clip_len = seq_cfg.clip_seq_len
    cf = video_info.clip_frames
    if cf.shape[0] < clip_len:
        pad = cf[-1:].expand(clip_len - cf.shape[0], *cf.shape[1:])
        video_info.clip_frames = torch.cat([cf, pad], dim=0)
        print(f"    [PAD] clip_frames {cf.shape[0]} -> {clip_len}")
    elif cf.shape[0] > clip_len:
        video_info.clip_frames = cf[:clip_len]
        print(f"    [TRIM] clip_frames {cf.shape[0]} -> {clip_len}")

    return video_info


def concat_videos(chunk_paths, output_path):
    """Sirali chunk listesini uc uca birlestir (ffmpeg concat demuxer).
    Ayni codec + cozunurluk = -c copy ile siyah frame olmaz."""
    list_file = output_path + '.txt'
    with open(list_file, 'w') as f:
        for p in chunk_paths:
            f.write(f"file '{p}'\n")
    result = subprocess.run([
        'ffmpeg', '-y', '-f', 'concat', '-safe', '0',
        '-i', list_file, '-c', 'copy', output_path
    ], capture_output=True, text=True)
    os.remove(list_file)
    if result.returncode != 0:
        raise ValueError(f"ffmpeg concat basarisiz: {result.stderr[-300:]}")


print("Yardimci fonksiyonlar tanimlandi.")

Yardimci fonksiyonlar tanimlandi.


## Kurulum

Şimdi MMAudio reposunu klonlayıp gerekli kütüphaneleri yükleyeceğiz.  
Bu adım birkaç dakika sürebilir.

In [5]:
# MMAudio reposunu klonla ve tum bagimliliklari kur
!git clone https://github.com/hkchengrex/MMAudio.git
%cd MMAudio
!pip install -e . -q

# Pip cache'i temizle (RAM tasarrufu)
!pip cache purge -q

print("\nKurulum tamamlandi!")

Cloning into 'MMAudio'...
remote: Enumerating objects: 685, done.
remote: Counting objects: 100% (300/300), done.
remote: Compressing objects: 100% (117/117), done.
remote: Total 685 (delta 235), reused 187 (delta 183), pack-reused 385 (from 2)
Receiving objects: 100% (685/685), 9.28 MiB | 1.10 MiB/s, done.
Resolving deltas: 100% (405/405), done.
/content/MMAudio
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.4/36.4 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 5.2 MB/s

In [6]:
# Model agirliklarini indir (Drive cache destekli)
import torch
from mmaudio.eval_utils import all_model_cfg
from huggingface_hub import hf_hub_download

model_cfg = all_model_cfg[MODEL_NAME]
nsfw_filename = "mmaudio_large_44k_nsfw_gold_8.5k_final_fp16.safetensors"

drive_model_path = f"/content/drive/MyDrive/{DRIVE_MODEL_FOLDER}"
local_model_dir = "/content/model_cache"
cache_marker = f"{drive_model_path}/.cache_complete"

use_cache = CACHE_MODELS_ON_DRIVE
cache_hit = use_cache and os.path.exists(cache_marker)

# --- Adim 1: Model dosyalarini hazirla ---

if cache_hit:
    # Drive'dan local'e kopyala (indirme atla)
    os.makedirs(local_model_dir, exist_ok=True)
    print(f"Drive/{DRIVE_MODEL_FOLDER}/ icinde model cache bulundu!")
    print("Modeller Drive'dan local'e kopyalaniyor...\n")
    for fname in os.listdir(drive_model_path):
        if fname.startswith('.'):
            continue
        src = f"{drive_model_path}/{fname}"
        if os.path.isfile(src):
            size_mb = os.path.getsize(src) / (1024**2)
            print(f"  {fname} ({size_mb:.0f} MB)")
            shutil.copy2(src, f"{local_model_dir}/{fname}")
    print("\nModeller cache'den yuklendi!")

if not cache_hit:
    # HuggingFace'den indir
    print("Modeller indiriliyor (ilk sefer, ~3 dk)...\n")
    model_cfg.download_if_needed()

# --- Adim 2: NSFW model yolunu belirle ---

if cache_hit:
    nsfw_model_path = f"{local_model_dir}/{nsfw_filename}"
else:
    nsfw_model_path = hf_hub_download(
        repo_id="phazei/NSFW_MMaudio",
        filename=nsfw_filename,
    )
    print(f"NSFW model indirildi: {nsfw_model_path}")

# --- Adim 3: Cache'e kaydet (sadece cache acik + ilk sefer) ---

if use_cache and not cache_hit:
    os.makedirs(drive_model_path, exist_ok=True)
    os.makedirs(local_model_dir, exist_ok=True)
    print(f"\nModeller Drive/{DRIVE_MODEL_FOLDER}/ klasorune kaydediliyor...")
    files_to_cache = [
        ('vae.pth', str(model_cfg.vae_path)),
        ('synchformer.pth', str(model_cfg.synchformer_ckpt)),
        (nsfw_filename, nsfw_model_path),
    ]
    if hasattr(model_cfg, 'bigvgan_16k_path') and model_cfg.bigvgan_16k_path:
        files_to_cache.append(('bigvgan_16k.pth', str(model_cfg.bigvgan_16k_path)))
    total_mb = 0
    for cache_name, src_path in files_to_cache:
        if src_path and os.path.exists(src_path):
            size_mb = os.path.getsize(src_path) / (1024**2)
            total_mb += size_mb
            print(f"  {cache_name} ({size_mb:.0f} MB)")
            shutil.copy2(src_path, f"{drive_model_path}/{cache_name}")
    Path(cache_marker).touch()
    print(f"Cache kaydedildi! (toplam: {total_mb:.0f} MB)")

# --- Adim 4: Cache hit ise model_cfg path'lerini local'e yonlendir ---

if cache_hit:
    model_cfg.vae_path = f"{local_model_dir}/vae.pth"
    model_cfg.synchformer_ckpt = f"{local_model_dir}/synchformer.pth"
    bigvgan_local = f"{local_model_dir}/bigvgan_16k.pth"
    if os.path.exists(bigvgan_local):
        model_cfg.bigvgan_16k_path = bigvgan_local

torch.cuda.empty_cache()
print(f"\nModel(ler) hazir! (NSFW FP16)")

Drive/mmaudio-models/ icinde model cache bulundu!
Modeller Drive'dan local'e kopyalaniyor...

  vae.pth (1165 MB)
  synchformer.pth (906 MB)
  mmaudio_large_44k_nsfw_gold_8.5k_final_fp16.safetensors (1965 MB)

Modeller cache'den yuklendi!

Model(ler) hazir! (NSFW FP16)


## Model Yükleme

Model ağırlıklarını GPU'ya yükler. Bu hücre bir kez çalıştırılır.  
Farklı ayarlarla tekrar denemek için sadece **Ses Üretimi** hücresini tekrar çalıştırın.

In [7]:
# Model yukle (bir kez calistirin)
import psutil
import torchaudio
from safetensors.torch import load_file
from mmaudio.eval_utils import setup_eval_logging, generate
from mmaudio.model.networks import get_my_mmaudio
from mmaudio.model.utils.features_utils import FeaturesUtils

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
setup_eval_logging()

device = 'cuda'
dtype = torch.float16  # T4 GPU bfloat16 desteklemez

def mem_status(label):
    ram = psutil.virtual_memory()
    gpu_used = torch.cuda.memory_allocated() / (1024**3)
    gpu_reserved = torch.cuda.memory_reserved() / (1024**3)
    print(f"[{label}] RAM: {ram.used/(1024**3):.1f}/{ram.total/(1024**3):.1f} GB (bos: {ram.available/(1024**3):.1f}) | GPU: {gpu_used:.1f} GB (reserved: {gpu_reserved:.1f})")

mem_status("baslangic")

# Agirliklari dogrudan GPU'ya yukle (CPU RAM kullanmadan)
print("Net olusturuluyor...")
net = get_my_mmaudio(model_cfg.model_name).to(device, dtype).eval()

# NSFW FP16 safetensors agirliklarini yukle
weights = load_file(nsfw_model_path, device=device)
net.load_weights(weights)
del weights
gc.collect()
torch.cuda.empty_cache()
mem_status("net GPU'da")

print("FeaturesUtils olusturuluyor...")
feature_utils = FeaturesUtils(
    tod_vae_ckpt=model_cfg.vae_path,
    synchformer_ckpt=model_cfg.synchformer_ckpt,
    enable_conditions=True,
    mode=model_cfg.mode,
    bigvgan_vocoder_ckpt=model_cfg.bigvgan_16k_path,
    need_vae_encoder=False
)
mem_status("FeaturesUtils CPU'da olusturuldu")

feature_utils = feature_utils.to(device, dtype).eval()
gc.collect()
torch.cuda.empty_cache()
mem_status("FeaturesUtils GPU'ya tasindi")

print("Model hazir! (NSFW FP16)")

[baslangic] RAM: 1.6/12.7 GB (bos: 10.7) | GPU: 0.0 GB (reserved: 0.0)
Net olusturuluyor...


INFO:root:Parsing model identifier. Schema: hf-hub, Identifier: apple/DFN5B-CLIP-ViT-H-14-384
[INFO    ]: Parsing model identifier. Schema: hf-hub, Identifier: apple/DFN5B-CLIP-ViT-H-14-384
INFO:root:Attempting to load config from HF Hub: apple/DFN5B-CLIP-ViT-H-14-384
[INFO    ]: Attempting to load config from HF Hub: apple/DFN5B-CLIP-ViT-H-14-384


[net GPU'da] RAM: 3.1/12.7 GB (bos: 9.2) | GPU: 2.0 GB (reserved: 2.1)
FeaturesUtils olusturuluyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
INFO:httpx:HTTP Request: HEAD https://huggingface.co/apple/DFN5B-CLIP-ViT-H-14-384/resolve/main/open_clip_config.json "HTTP/1.1 307 Temporary Redirect"
[INFO    ]: HTTP Request: HEAD https://huggingface.co/apple/DFN5B-CLIP-ViT-H-14-384/resolve/main/open_clip_config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/apple/DFN5B-CLIP-ViT-H-14-378/resolve/main/open_clip_config.json "HTTP/1.1 307 Temporary Redirect"
[INFO    ]: HTTP Request:

open_clip_config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/nvidia/bigvgan_v2_44khz_128band_512x/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[INFO    ]: HTTP Request: HEAD https://huggingface.co/nvidia/bigvgan_v2_44khz_128band_512x/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nvidia/bigvgan_v2_44khz_128band_512x/95a9d1dcb12906c03edd938d77b9333d6ded7dfb/config.json "HTTP/1.1 200 OK"
[INFO    ]: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/nvidia/bigvgan_v2_44khz_128band_512x/95a9d1dcb12906c03edd938d77b9333d6ded7dfb/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/nvidia/bigvgan_v2_44khz_128band_512x/resolve/main/bigvgan_generator.pt "HTTP/1.1 302 Found"
[INFO    ]: HTTP Request: HEAD https://huggingface.co/nvidia/bigvgan_v2_44khz_128band_512x/resolve/main/bigvgan_generator.pt "HTTP/1.1 302 Found"
[WARNING ]: Warning: You are sending 

Loading weights from nvidia/bigvgan_v2_44khz_128band_512x


bigvgan_generator.pt:   0%|          | 0.00/489M [00:00<?, ?B/s]

Removing weight norm...
[FeaturesUtils CPU'da olusturuldu] RAM: 9.7/12.7 GB (bos: 2.6) | GPU: 2.0 GB (reserved: 2.1)
[FeaturesUtils GPU'ya tasindi] RAM: 9.6/12.7 GB (bos: 2.8) | GPU: 4.6 GB (reserved: 4.8)
Model hazir! (NSFW FP16)


## Ses Üretimi

Videolardan ses üretip, ffmpeg ile birleştirip Drive'a kaydeder.  
Ayarları değiştirdikten sonra bu hücreyi tekrar çalıştırmanız yeterlidir.

In [ ]:
# Ses uret + video ile birlestir + Drive'a kaydet (klasor yapisi korunur)
from mmaudio.model.flow_matching import FlowMatching
from mmaudio.eval_utils import load_video

fm = FlowMatching(min_sigma=0, inference_mode=INFERENCE_MODE, num_steps=NUM_STEPS)
rng = torch.Generator(device=device)
seq_cfg = model_cfg.seq_cfg

video_list = video_items if BATCH_MODE else [video_items[VIDEO_INDEX]]
print(f"Islenecek video sayisi: {len(video_list)}")
print("=" * 50)
mem_status("ses uretimi baslangici")

skipped = 0
errors = 0

for i, (rel_dir, filename) in enumerate(video_list):
    stem = os.path.splitext(filename)[0]
    display_name = f"{rel_dir}/{filename}" if rel_dir else filename
    src_path = make_drive_path(drive_video_path, rel_dir, filename)
    out_dir = make_drive_path(drive_output_path, rel_dir)
    output_name = f"{stem}_audio.mp4"
    drive_output_file = f"{out_dir}/{output_name}"

    print(f"\n[{i+1}/{len(video_list)}] {display_name}")

    # Zaten islenmis mi kontrol et
    if os.path.exists(drive_output_file):
        out_display = f"{rel_dir}/{output_name}" if rel_dir else output_name
        print(f"  ATLANDI: Drive/{DRIVE_OUTPUT_FOLDER}/{out_display} zaten mevcut")
        skipped += 1
        continue

    # Sentinel degiskenler (finally'da guvenli temizlik icin)
    clip_v = sync_v = audios = audio = video_info = None
    local_video_orig = f"/content/{i}_{filename}"
    local_video_720p = f"/content/{i}_{stem}_720p.mp4"
    local_audio = f"/content/{i}_{stem}_audio.flac"
    local_output = f"/content/{i}_{stem}_audio.mp4"
    local_files = [local_video_orig, local_video_720p, local_audio, local_output]
    temp_dir = f"/content/temp_chunks/{i}_{stem}"

    try:
        # 1. Drive'dan local'e kopyala
        shutil.copy2(src_path, local_video_orig)

        # 2. Sure al + chunk hesapla (720p'den ONCE)
        duration = get_video_duration(local_video_orig)
        chunks = calculate_chunks(duration, MAX_CHUNK_DURATION)

        if len(chunks) == 1:
            # =============================================
            #  KISA VIDEO — mevcut akis aynen
            # =============================================
            resize_video_720p(local_video_orig, local_video_720p)
            local_video = local_video_720p
            orig_mb = os.path.getsize(local_video_orig) / (1024**2)
            new_mb = os.path.getsize(local_video) / (1024**2)
            print(f"  720p ({orig_mb:.0f} MB -> {new_mb:.0f} MB)")

            print(f"  Sure: {duration:.2f}s")
            seq_cfg.duration = duration
            net.update_seq_lengths(seq_cfg.latent_seq_len, seq_cfg.clip_seq_len, seq_cfg.sync_seq_len)

            video_info = load_video(Path(local_video), duration)
            video_info = fix_video_frames(video_info, seq_cfg)
            clip_v = video_info.clip_frames.unsqueeze(0).to(device, dtype)
            sync_v = video_info.sync_frames.unsqueeze(0).to(device, dtype)
            video_info = None
            gc.collect()
            mem_status("load_video sonrasi")

            rng.manual_seed(SEED)
            print(f"  Uretiliyor... (prompt: '{PROMPT or '(bos)'}')")
            with torch.inference_mode():
                audios = generate(
                    clip_video=clip_v, sync_video=sync_v,
                    text=[PROMPT or ''], negative_text=[NEGATIVE_PROMPT],
                    feature_utils=feature_utils, net=net, fm=fm,
                    rng=rng, cfg_strength=CFG_STRENGTH,
                )
            audio = audios.float().cpu()[0]
            clip_v = sync_v = audios = None
            gc.collect()
            torch.cuda.empty_cache()
            mem_status("generate sonrasi")

            torchaudio.save(local_audio, audio, seq_cfg.sampling_rate)
            audio = None
            merge_video_audio(local_video_orig, local_audio, local_output)
            os.makedirs(out_dir, exist_ok=True)
            shutil.copy2(local_output, drive_output_file)
            out_display = f"{rel_dir}/{output_name}" if rel_dir else output_name
            print(f"  -> Drive/{DRIVE_OUTPUT_FOLDER}/{out_display}")

        else:
            # =============================================
            #  UZUN VIDEO — chunk pipeline
            # =============================================
            print(f"  Uzun video: {duration:.1f}s -> {len(chunks)} parca")
            os.makedirs(temp_dir, exist_ok=True)
            merged_chunk_paths = []  # sirali liste — concat icin

            for ci, (start, dur) in enumerate(chunks):
                chunk_label = f"part{ci+1:03d}"
                chunk_720p = f"{temp_dir}/{stem}_{chunk_label}.mp4"
                chunk_audio = f"{temp_dir}/{stem}_{chunk_label}.flac"
                chunk_merged = f"{temp_dir}/{stem}_{chunk_label}_merged.mp4"

                print(f"  [{ci+1}/{len(chunks)}] {chunk_label} ({start:.1f}s-{start+dur:.1f}s, {dur:.1f}s)")

                # a) 720p chunk cikar (resize + split tek komutta)
                extract_chunk_720p(local_video_orig, start, dur, chunk_720p)

                # b) Model sequence guncelle
                seq_cfg.duration = dur
                net.update_seq_lengths(seq_cfg.latent_seq_len, seq_cfg.clip_seq_len, seq_cfg.sync_seq_len)

                # c) Video yukle + ses uret
                video_info = load_video(Path(chunk_720p), dur)
                video_info = fix_video_frames(video_info, seq_cfg)
                clip_v = video_info.clip_frames.unsqueeze(0).to(device, dtype)
                sync_v = video_info.sync_frames.unsqueeze(0).to(device, dtype)
                video_info = None
                gc.collect()

                rng.manual_seed(SEED)
                print(f"    Uretiliyor... (prompt: '{PROMPT or '(bos)'}')")
                with torch.inference_mode():
                    audios = generate(
                        clip_video=clip_v, sync_video=sync_v,
                        text=[PROMPT or ''], negative_text=[NEGATIVE_PROMPT],
                        feature_utils=feature_utils, net=net, fm=fm,
                        rng=rng, cfg_strength=CFG_STRENGTH,
                    )
                audio = audios.float().cpu()[0]

                # d) Kaydet + merge (720p chunk + audio)
                torchaudio.save(chunk_audio, audio, seq_cfg.sampling_rate)
                merge_video_audio(chunk_720p, chunk_audio, chunk_merged)
                merged_chunk_paths.append(chunk_merged)

                # e) Temizlik — sadece merged kalir
                clip_v = sync_v = audios = audio = video_info = None
                gc.collect()
                torch.cuda.empty_cache()
                cleanup_files(chunk_720p, chunk_audio)
                mem_status(f"  chunk {ci+1}/{len(chunks)} temizlendi")

            # f) Tum parcalari birlestir
            print(f"  {len(merged_chunk_paths)} parca birlestiriliyor...")
            concat_videos(merged_chunk_paths, local_output)

            # g) Drive'a yukle + temp temizle
            os.makedirs(out_dir, exist_ok=True)
            shutil.copy2(local_output, drive_output_file)
            shutil.rmtree(temp_dir, ignore_errors=True)
            out_display = f"{rel_dir}/{output_name}" if rel_dir else output_name
            print(f"  -> Drive/{DRIVE_OUTPUT_FOLDER}/{out_display}")

    except Exception as e:
        print(f"  HATA: {e}")
        errors += 1

    finally:
        # Sentinel'ler sayesinde del her zaman guvenli
        del clip_v, sync_v, audios, audio, video_info
        gc.collect()
        torch.cuda.empty_cache()
        cleanup_files(*local_files)
        if os.path.exists(temp_dir):
            shutil.rmtree(temp_dir, ignore_errors=True)
        mem_status(f"video {i+1} temizlendi")

# Ozet
total = len(video_list)
success = total - skipped - errors
print("\n" + "=" * 50)
print(f"Tamamlandi! {total} video ({success} basarili, {skipped} atlandi, {errors} hata)")
drive.flush_and_unmount()
print(f"Sonuclar: Drive/{DRIVE_OUTPUT_FOLDER}/")

Islenecek video sayisi: 151
[ses uretimi baslangici] RAM: 9.6/12.7 GB (bos: 2.8) | GPU: 4.6 GB (reserved: 4.8)

[1/151] hafta1/1/1.mp4
  HATA: ffmpeg resize basarisiz: 58. 76.100
  libavdevice    58. 13.100 / 58. 13.100
  libavfilter     7.110.100 /  7.110.100
  libswscale      5.  9.100 /  5.  9.100
  libswresample   3.  9.100 /  3.  9.100
  libpostproc    55.  9.100 / 55.  9.100
Unrecognized option 'fps_mode'.
Error splitting the argument list: Option not found

[video 1 temizlendi] RAM: 9.5/12.7 GB (bos: 2.8) | GPU: 4.6 GB (reserved: 4.8)

[2/151] hafta1/1/450x_auto__ (10).mp4
  Uzun video: 14.1s -> 2 parca
  [1/2] part001 (0.0s-7.0s, 7.0s)
  HATA: ffmpeg chunk extract basarisiz: 58. 76.100
  libavdevice    58. 13.100 / 58. 13.100
  libavfilter     7.110.100 /  7.110.100
  libswscale      5.  9.100 /  5.  9.100
  libswresample   3.  9.100 /  3.  9.100
  libpostproc    55.  9.100 / 55.  9.100
Unrecognized option 'fps_mode'.
Error splitting the argument list: Option not found

[video 